# CSD Data Manipulation - Category System

## How to Add New Categories

### Step 1: Update your tariff-hscodes CSV
Add entries with your new category names:
```csv
HS Code,Category
440710,Lumber (old)
440910,Lumber (new)
...
```

### Step 2: Update CATEGORY_CONFIG (Cell 3)
Map each CSV category name to a short column prefix:
```python
CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Lumber (old)': 'LumOld',   # <-- Add new categories
    'Lumber (new)': 'LumNew',   # <-- Add new categories
    ...
}
```

### Step 3: Run all cells
The notebook will automatically:
- Create NAICS sets for each category
- Generate `{Prefix}_B`, `{Prefix}_E` columns (weighted business/employee counts)
- Generate `{Prefix}_1`, `{Prefix}_2`, `{Prefix}_3` columns (percentages for choropleth)
- Export everything to GeoJSON, Shapefile, and CSV

### Output Columns per Category
- `{Prefix}_B` = Weighted number of businesses
- `{Prefix}_E` = Weighted number of employees (by work location)
- `{Prefix}_C` = Weighted number of employees (by residence)
- `{Prefix}_1` = % of all businesses affected
- `{Prefix}_2` = % of all employees affected (by work location)
- `{Prefix}_3` = % of census population in affected jobs (by residence)

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.ops import unary_union
from tqdm import tqdm
import json
import gc
import time
from datetime import timedelta

## Configuration: Define your tariff categories here

In [ ]:
# ============================================================
# CATEGORY CONFIGURATION - Edit this to add/modify categories
# ============================================================

# Path to the tariff-hscodes CSV file
TARIFF_HSCODES_FILE = 'tariff-hscodes_noderiv.csv'

# Category configuration: maps CSV category names to short column prefixes
# Format: 'Category Name in CSV': 'Short_Prefix_for_Columns'

CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Steel': 'Steel',
    'Copper': 'Cop',
    'Energy Mineral': 'Ene',
    'MHDV': 'MHDV',
    # Add new categories here:
    'Lumber (old)': 'LumOld',
    'Lumber (new)': 'LumNew',
}

# Always-included special categories (don't change these unless you know what you're doing)
SPECIAL_CATEGORIES = ['nonCUSMA', 'Total']  # nonCUSMA = goods not covered by CUSMA

print(f"✅ Configured {len(CATEGORY_CONFIG)} tariff categories: {list(CATEGORY_CONFIG.keys())}")
print(f"   Using tariff file: {TARIFF_HSCODES_FILE}")

✅ Configured 9 tariff categories: ['Auto', 'Aluminum', 'Steel', 'Copper', 'Lumber', 'Energy Mineral', 'MHDV', 'Lumber (old)', 'Lumber (new)']
   Using tariff file: tariff-hscodes_noderiv.csv


# STEP 3: Connecting Tariffed HS Codes, NAICS Codes and respective CUSMA Non-Utilisation Rates via Concordance Table

In [3]:
tariffed = pd.read_csv(TARIFF_HSCODES_FILE, encoding_errors='ignore', dtype={'HS Code': str})

tariffed['HS_Code_6digit'] = (
    tariffed['HS Code']
    .str.replace('.', '', regex=False)
    .str[:6]
)

tariffed = tariffed[['HS_Code_6digit', 'Category']].drop_duplicates()

# Validate that all categories in CSV are in our config
csv_categories = set(tariffed['Category'].dropna().unique())
configured_categories = set(CATEGORY_CONFIG.keys())
unknown_categories = csv_categories - configured_categories

if unknown_categories:
    print(f"⚠️ WARNING: Found categories in CSV not in CATEGORY_CONFIG: {unknown_categories}")
    print("   Add them to CATEGORY_CONFIG or they will be treated as 'nonCUSMA'")
else:
    print(f"✅ All CSV categories are configured: {csv_categories}")

✅ All CSV categories are configured: {'Steel', 'Lumber (new)', 'MHDV', 'Copper', 'Aluminum', 'Lumber (old)', 'Auto', 'Energy Mineral'}


In [4]:
concordance = pd.read_csv('C616_HS8toNaics6_concord_202505.csv', dtype={'hts10': str})

concordance['HS_Code_6digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:6]
)

concordance['HS_Code_2digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:2]
)

concordance['NAICS'] = concordance['NAICS 6 Code'].astype(str)

concordance = concordance[['HS_Code_6digit', 'NAICS', 'HS_Code_2digit']].drop_duplicates()

print(concordance[concordance['HS_Code_6digit'] == '440311'].head())

     HS_Code_6digit   NAICS HS_Code_2digit
4468         440311  321114             44


In [5]:
util = pd.read_csv('USMCA Utilization Data.csv')

util['HS_Code_2digit'] = (
    util['HS Classification']
    .astype(str)
    .str[:2]
)

util['nonutil_rate'] = util['USMCA_Nonutilisation_May2025']

util = util[['HS_Code_2digit', 'nonutil_rate']]

Setting the non-utilisation rate for those with sectoral tariffs at 1 reflects the fact that the 35% tariffs on non-CUSMA goods does not apply to the sectoral tariffs and that producers impacted by sectoral tariffs cannot use CUSMA to get their goods tariff-free.

In [6]:
naics_tar = concordance.merge(tariffed, on='HS_Code_6digit', how='left')

counts = naics_tar['HS_Code_6digit'].value_counts().reset_index()
naics_tarc = naics_tar.merge(counts, on='HS_Code_6digit', how='left')

naics_imp = naics_tarc.merge(util, on='HS_Code_2digit', how='left')

# Making the codes with a sectoral tariff category assigned to have a non-util rate value of 1
# This ensures that when multiplied later, sectoral tariffs do not affect the CUSMA non-utilisation rates to compute the impact of nonCUSMA 35% tariffs
naics_imp.loc[naics_imp['Category'].notna(), 'nonutil_rate'] = 1
naics_imp['Category'] = naics_imp['Category'].fillna('nonCUSMA')
naics_imp

,HS_Code_6digit,NAICS,HS_Code_2digit,Category,count,nonutil_rate
0,010110,112920,01,nonCUSMA,1,0.42
1,010121,112920,01,nonCUSMA,1,0.42
2,010129,112920,01,nonCUSMA,1,0.42
3,010130,112920,01,nonCUSMA,1,0.42
4,010190,112920,01,nonCUSMA,1,0.42
...,...,...,...,...,...,...
6980,961620,314990,96,nonCUSMA,1,0.84
6981,961700,332439,96,nonCUSMA,1,0.84
6982,961800,339990,96,nonCUSMA,1,0.84
6983,961900,322291,96,nonCUSMA,1,0.84


# STEP 4: Getting Weights by Province/Territory

Since multiple NAICS codes may contribute to the production of one HS code product, and we do not how much of a part does each NAICS contribute to the whole HS code good production, **thus an assumption is made to divide them equally**. Hence, when each export value is added, it is divided them by the count (how many times does that HS Code get repeated).  

Meanwhile, the non-utilisation rate of CUSMA exemption by each HS Code is first multiplied to the total value of each HS Code export to the US, before divided by the count.

In [7]:
# Initialize the DataFrame with NAICS data
tariff_exp_val = naics_imp.copy()

# Define all regions to process
provinces = ['NL', 'PEI', 'NS', 'NB', 'QC', 'ON', 'MB', 'SK', 'AL', 'BC', 'YK', 'NWT', 'NU']
cols = ['Commodity', 'Value ($)']

for province in provinces:
    # Process Global data
    global_df = pd.read_csv(f'{province}-Global.csv', usecols=cols)
    global_df['HS_Code_6digit'] = global_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    global_df[f'{province}_Global'] = global_df['Value ($)']
    global_df = global_df[['HS_Code_6digit', f'{province}_Global']]
    
    tariff_exp_val = tariff_exp_val.merge(global_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_Global'] = tariff_exp_val[f'{province}_Global'] / tariff_exp_val['count']
    
    # Process US data
    us_df = pd.read_csv(f'{province}-US.csv', usecols=cols)
    us_df['HS_Code_6digit'] = us_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    us_df[f'{province}_US'] = us_df['Value ($)']
    us_df = us_df[['HS_Code_6digit', f'{province}_US']]
    
    tariff_exp_val = tariff_exp_val.merge(us_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_US'] = (tariff_exp_val[f'{province}_US'] * tariff_exp_val['nonutil_rate']) / tariff_exp_val['count']

# Drop the non-relevant columns
tariff_exp_val = tariff_exp_val.drop(columns=['HS_Code_2digit', 'Category', 'count', 'nonutil_rate'])

tariff_exp_val = tariff_exp_val.fillna(0)

tariff_exp_val

,HS_Code_6digit,NAICS,NL_Global,NL_US,PEI_Global,PEI_US,NS_Global,NS_US,NB_Global,NB_US,...,AL_Global,AL_US,BC_Global,BC_US,YK_Global,YK_US,NWT_Global,NWT_US,NU_Global,NU_US
0,010110,112920,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
1,010121,112920,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,102006.0,42842.52,43608.0,18315.36,0.0,0.0,0.0,0.0,0.0,0.0
2,010129,112920,0.0,0.0,103341.0,43403.22,767570.0,322379.40,245243.0,103002.06,...,38107217.0,6846298.20,8940055.0,3722693.10,0.0,0.0,0.0,0.0,0.0,0.0
3,010130,112920,0.0,0.0,0.0,0.00,0.0,0.00,5591.0,2348.22,...,0.0,0.00,16453.0,6910.26,0.0,0.0,0.0,0.0,0.0,0.0
4,010190,112920,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,0.0,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6980,961620,314990,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,0.0,0.00,26374.0,19699.68,0.0,0.0,0.0,0.0,0.0,0.0
6981,961700,332439,0.0,0.0,0.0,0.00,9.0,0.00,0.0,0.00,...,4454.0,3741.36,191510.0,58298.52,0.0,0.0,0.0,0.0,0.0,0.0
6982,961800,339990,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,...,0.0,0.00,57388.0,30522.24,0.0,0.0,0.0,0.0,0.0,0.0
6983,961900,322291,0.0,0.0,0.0,0.00,0.0,0.00,113029441.0,94496776.08,...,10736.0,0.00,408726.0,4863.60,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# Step 1: Aggregate ONLY the necessary sums (Global/US columns)
aggregates = {
    **{f'{province}_Global': (f'{province}_Global', 'sum') for province in provinces},
    **{f'{province}_US': (f'{province}_US', 'sum') for province in provinces},
}

weight_naics = tariff_exp_val.groupby('NAICS', as_index=False).agg(**aggregates)

# Step 2: Compute rates and keep ONLY those columns
for province in provinces:
    global_col = f'{province}_Global'
    us_col = f'{province}_US'
    rate_col = f'{province}'
    
    weight_naics[rate_col] = (
        weight_naics[us_col] / weight_naics[global_col]
    )

# Step 3: Select only naics, naics_2digit, and rate columns
final_columns = ['NAICS'] + [f'{province}' for province in provinces]
weight_naics = weight_naics[final_columns]
weight_naics = weight_naics.fillna(0)

# Result
weight_naics

,NAICS,NL,PEI,NS,NB,QC,ON,MB,SK,AL,BC,YK,NWT,NU
0,111110,0.000000,0.000000,0.000000,0.620000,0.016695,0.057076,0.064938,0.000728,0.024138,0.000000,0.00,0.0,0.00
1,111120,0.620000,0.620000,0.000000,0.124930,0.039934,0.371635,0.094353,0.047928,0.034204,0.028733,0.00,0.0,0.00
2,111130,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.0,0.00
3,111140,0.000000,0.000000,0.620000,0.620000,0.045639,0.032567,0.030494,0.038389,0.032376,0.028748,0.00,0.0,0.00
4,111150,0.000000,0.000000,0.000000,0.000000,0.194542,0.074227,0.370000,0.000000,0.370000,0.243865,0.00,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,339920,0.079019,0.227019,0.706480,0.243381,0.808154,0.653166,0.582863,0.866177,0.663320,0.587708,1.00,0.0,0.00
306,339930,0.358952,0.000000,0.397013,0.420000,0.347140,0.248730,0.337235,0.409485,0.368632,0.255254,0.00,0.0,0.42
307,339940,0.000000,0.000000,0.049719,0.831198,0.467514,0.732703,0.859608,0.150000,0.720095,0.691042,0.00,0.0,0.00
308,339950,0.000000,0.150000,0.149934,0.699527,0.748507,0.562703,0.151730,0.604759,0.206830,0.265313,0.15,0.0,0.15


# STEP 5: Using filtered NAICS Codes to filter directly-impacted businesses and estimate directly-impacted employees

In [9]:
# DYNAMICALLY CREATE NAICS SETS FOR EACH CATEGORY

# Dictionary to store NAICS codes for each category
category_naics = {}

# Create NAICS sets for each configured category
for csv_name, short_name in CATEGORY_CONFIG.items():
    category_naics[short_name] = set(
        naics_imp[naics_imp['Category'] == csv_name]['NAICS'].unique()
    )
    print(f"  {short_name}: {len(category_naics[short_name])} NAICS codes")

# Special categories (always included)
category_naics['CUSMA'] = set(naics_imp[naics_imp['Category'] == 'nonCUSMA']['NAICS'].unique())
category_naics['Total'] = set(naics_imp['NAICS'].unique())

print(f"\n✅ Created {len(category_naics)} NAICS sets")
print(f"   Total unique NAICS codes: {len(category_naics['Total'])}")

# For backward compatibility, also create individual variables (optional)
total_naics = category_naics['Total']

  Auto: 38 NAICS codes
  Alum: 47 NAICS codes
  Steel: 43 NAICS codes
  Cop: 6 NAICS codes
  Lum: 0 NAICS codes
  Ene: 39 NAICS codes
  MHDV: 5 NAICS codes
  LumOld: 5 NAICS codes
  LumNew: 12 NAICS codes

✅ Created 11 NAICS sets
   Total unique NAICS codes: 310


In [10]:
# ============================================================
# DYNAMICALLY CREATE COLUMN STRUCTURE
# ============================================================

# Base columns (always present)
col_i = ['DA', 'All_Businesses', 'All_Employees']

# Add columns for each category (Business and Employee counts)
all_category_prefixes = list(CATEGORY_CONFIG.values()) + ['CUSMA', 'Total']
for prefix in all_category_prefixes:
    col_i.extend([f'{prefix}_B', f'{prefix}_E'])

# Add individual NAICS codes as column headers for Est_Employees by NAICS
col_i.extend(sorted(total_naics))

business = pd.DataFrame(columns = col_i)

print(f"✅ Created DataFrame with {len(col_i)} columns")
print(f"   Category columns: {[p for p in all_category_prefixes]}")

✅ Created DataFrame with 335 columns
   Category columns: ['Auto', 'Alum', 'Steel', 'Cop', 'Lum', 'Ene', 'MHDV', 'LumOld', 'LumNew', 'CUSMA', 'Total']


In [11]:
# ============================================================
# MAIN PROCESSING LOOP - DYNAMIC CATEGORIES
# ============================================================

chunk_size = 1_000_000
province_code = {10:'NL',11:'PEI',12:'NS',13:'NB',24:'QC',35:'ON',46:'MB',47:'SK',48:'AL',59:'BC',60:'YK',61:'NWT',62:'NU'}

# melting weights in long form to prepare for a vectorized merge
wlong = (
    weight_naics
    .melt(id_vars='NAICS', var_name='Province', value_name='Rate')
)

# loading necessary data
all_cols = pd.read_csv('input-data/large_size_data/Dec2022_Estabcounts_byDA.csv', encoding='ISO-8859-1', nrows=1).columns
cols_to_keep = [c for c in all_cols if c != 'Without employees']

# suggesting dtypes for performance purposes
dtype_hint = {
    '1-4':'Int64','5-9':'Int64','10-19':'Int64','20-49':'Int64',
    '50-99':'Int64','100-199':'Int64','200-499':'Int64','500 +':'Int64',
    'Total, with employees':'Int64',
}

total_start = time.time()
chunk_num = 0

# Preparing two empty lists
agg_frames = []       # List for weighted amount of businesses and est employees for each tariff
per_naics_frames = [] # List for total amount of employees for each NAICS code in each ADA --> needed for Step 8 later

# Build the list of aggregation columns dynamically
agg_cols = ['All_Businesses', 'All_Employees']
for prefix in all_category_prefixes:
    agg_cols.extend([f'{prefix}_B', f'{prefix}_E'])

for chunk in pd.read_csv(
        'input-data/large_size_data/Dec2022_Estabcounts_byDA.csv',
        encoding='ISO-8859-1',
        chunksize=chunk_size,
        usecols=cols_to_keep,
        dtype=dtype_hint,
    ):
    t0 = time.time()
    chunk_num += 1

    # 1) Filter non-relevant rows
    chunk = chunk[~chunk['NAICS'].isin(['Sub-total, classified', 'Unclassified', 'Total'])].copy()

    # 2) Basic transforms (vectorized)
    # keep NAICS 6-digit as string
    chunk['NAICS'] = chunk['NAICS'].astype(str).str[:6]
    chunk['Business_per_NAICS'] = chunk['Total, with employees'].fillna(0)

    # estimate employees (vectorized)
    chunk['Est_Employees'] = (
        chunk['1-4'].fillna(0) * 3  +
        chunk['5-9'].fillna(0) * 7  +
        chunk['10-19'].fillna(0) * 15 +
        chunk['20-49'].fillna(0) * 35 +
        chunk['50-99'].fillna(0) * 75 +
        chunk['100-199'].fillna(0) * 150 +
        chunk['200-499'].fillna(0) * 350 +
        chunk['500 +'].fillna(0) * 550
    )

    # Province lookup
    # if 'DisseminationAre' isn’t numeric, ensure this still works (it uses first two chars)
    chunk['ProvinceCode'] = chunk['DisseminationAre'].astype(str).str[:2].astype(int, errors='ignore')
    chunk['Province'] = pd.Series(chunk['ProvinceCode']).map(province_code)

    # 3) Merge the per-(NAICS, Province) Rate (vectorized, no apply)
    merged = chunk.merge(wlong, how='left', on=['NAICS','Province'])

    # 4) Weighted columns (vectorized)
    merged['Weighted_Business']  = np.ceil(merged['Business_per_NAICS'] * merged['Rate'])
    merged['Weighted_Employees'] = np.ceil(merged['Est_Employees'] * merged['Rate'])

    # 5) DYNAMIC CATEGORY MASKS - Create masks for each category
    s = merged['NAICS']
    wb = merged['Weighted_Business']
    we = merged['Weighted_Employees']
    
    # Apply masks dynamically for each category
    for prefix, naics_set in category_naics.items():
        is_in_category = s.isin(naics_set) if len(naics_set) else pd.Series(False, index=s.index)
        merged[f'{prefix}_B'] = np.where(is_in_category, wb, 0)
        merged[f'{prefix}_E'] = np.where(is_in_category, we, 0)

    # Always aggregate the unweighted totals too
    merged['All_Businesses'] = merged['Business_per_NAICS']
    merged['All_Employees']  = merged['Est_Employees']

    # 6) Chunk-level aggregation in ONE groupby
    by_da = merged.groupby('DisseminationAre', as_index=False)[agg_cols].sum()

    # 7) Generating the data for second list --> number of jobs per NAICS in each DA
    is_total = s.isin(total_naics)
    per_naics = (
        merged.loc[is_total, ['DisseminationAre','NAICS','Est_Employees']]
        .pivot_table(index='DisseminationAre', columns='NAICS', values='Est_Employees',
                     aggfunc='sum', fill_value=0)
        .reset_index()
    )

    # Saving the data into the two different lists in each chunk
    agg_frames.append(by_da)
    per_naics_frames.append(per_naics)

    print(f"Chunk {chunk_num} processed in {time.time()-t0:.2f} sec")

# Combine all chunks together to form one big dataframe
agg_all = pd.concat(agg_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

per_naics_all = pd.concat(per_naics_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

business = agg_all.merge(per_naics_all, on='DisseminationAre', how='left')

business = business.rename(columns={'DisseminationAre':'DA'})

print(f"\n✅ All chunks processed in {time.time()-total_start:.2f} sec")
print(f"   Processed categories: {list(category_naics.keys())}")

business

Chunk 1 processed in 3.70 sec
Chunk 2 processed in 3.83 sec
Chunk 3 processed in 3.69 sec
Chunk 4 processed in 3.74 sec
Chunk 5 processed in 3.78 sec
Chunk 6 processed in 3.92 sec
Chunk 7 processed in 3.83 sec
Chunk 8 processed in 3.78 sec
Chunk 9 processed in 3.82 sec
Chunk 10 processed in 3.83 sec
Chunk 11 processed in 3.81 sec
Chunk 12 processed in 3.85 sec
Chunk 13 processed in 3.79 sec
Chunk 14 processed in 3.93 sec
Chunk 15 processed in 3.82 sec
Chunk 16 processed in 3.85 sec
Chunk 17 processed in 3.85 sec
Chunk 18 processed in 3.85 sec
Chunk 19 processed in 3.78 sec
Chunk 20 processed in 3.80 sec
Chunk 21 processed in 4.17 sec
Chunk 22 processed in 3.94 sec
Chunk 23 processed in 3.79 sec
Chunk 24 processed in 3.81 sec
Chunk 25 processed in 3.81 sec
Chunk 26 processed in 3.77 sec
Chunk 27 processed in 3.85 sec
Chunk 28 processed in 3.81 sec
Chunk 29 processed in 3.86 sec
Chunk 30 processed in 3.80 sec
Chunk 31 processed in 3.79 sec
Chunk 32 processed in 3.78 sec
Chunk 33 processe

,DA,All_Businesses,All_Employees,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,Cop_B,...,337215,337910,337920,339110,339910,339920,339930,339940,339950,339990
0,10000000,170,1006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,10010165,22,266,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,10010166,2,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,10010167,6,22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,10010168,5,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55246,62080023,2,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
55247,62080024,11,113,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
55248,62080025,11,356,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
55249,62080026,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


# STEP 6: Regrouping filtered data into CSDs

In [12]:
# Read DA polygons (KEEP geometry for spatial join)
da = gpd.read_file('input-data/large_size_data/lda_000b21a_e.shp')
da['DA'] = da['DAUID']
da['DADGUID'] = da['DGUID']
da = da[['DA', 'DADGUID', 'geometry']].copy() # geometry here from the shapefile?

# Read CSD polygons (these DGUIDs match your csd_neighbors IDs: 2021A0005...)
csd = gpd.read_file('../data/census/lcsd000b21a_e/lcsd000b21a_e.shp')
csdc = csd.copy()
csdc['CSDDGUID'] = csdc['DGUID']
csdc = csdc[['CSDDGUID', 'LANDAREA', 'geometry']].copy()

# Ensure both layers are in a projected CRS for spatial operations
da = da.to_crs('EPSG:3347')
csdc = csdc.to_crs('EPSG:3347')

print('DA rows:', len(da), '| CSD rows:', len(csdc))

DA rows: 57932 | CSD rows: 5161


In [13]:
# Build DA -> CSD relation via spatial containment (robust; no DGUID parsing)
da_pts = da[['DA', 'DADGUID', 'geometry']].copy()
da_pts['geometry'] = da_pts.geometry.representative_point()

# Spatial join: DA point within CSD polygon
da_relation = gpd.sjoin(
    da_pts,
    csdc[['CSDDGUID', 'LANDAREA', 'geometry']],
    how='left',
    predicate='within'
).drop(columns=['index_right', 'geometry'])  # keep attributes only; drop DA point geometry

# Match downstream expectations
da_relation['DA'] = pd.to_numeric(da_relation['DA'], errors='coerce').astype('Int64')

match_rate = da_relation['CSDDGUID'].notna().mean()
print(f"DA->CSD match rate (spatial): {match_rate:.3%}")

da_relation.head()

DA->CSD match rate (spatial): 100.000%


,DA,DADGUID,CSDDGUID,LANDAREA
0,10010165,2021S051210010165,2021A00051001519,446.0185
1,10010166,2021S051210010166,2021A00051001519,446.0185
2,10010167,2021S051210010167,2021A00051001519,446.0185
3,10010168,2021S051210010168,2021A00051001519,446.0185
4,10010169,2021S051210010169,2021A00051001519,446.0185


In [14]:
# ============================================================
# AGGREGATE TO CSD LEVEL - DYNAMIC CATEGORIES
# ============================================================

business_merged = (
    business.merge(da_relation, on='DA', how='right')
) 

# Separate numeric columns from geometry
numeric_cols = [col for col in business.columns if col != 'DA']

# Fill NA only for numeric columns, then cast to int64
for col in numeric_cols:
    business_merged[col] = business_merged[col].fillna(0).astype('int64')

# Build aggregation dictionary dynamically
agg_dict = {
    'All_Businesses': ('All_Businesses', 'sum'),
    'All_Employees': ('All_Employees', 'sum'),
    'LANDAREA': ('LANDAREA', 'first')
}

# Add category columns dynamically
for prefix in all_category_prefixes:
    agg_dict[f'{prefix}_B'] = (f'{prefix}_B', 'sum')
    agg_dict[f'{prefix}_E'] = (f'{prefix}_E', 'sum')

# Add dynamic aggregation rules for each NAICS code
for naics in total_naics:
    agg_dict[naics] = (naics, 'sum')

# Perform grouped aggregation
business_grouped = business_merged.groupby('CSDDGUID', as_index=False).agg(**agg_dict)

# Attach original CSD polygons (not simplified) for downstream outputs
business_grouped = business_grouped.merge(csdc[['CSDDGUID', 'geometry']], on='CSDDGUID', how='left')

print(f"✅ Aggregated to {len(business_grouped)} CSDs with {len(all_category_prefixes)} category pairs")

business_grouped

C:\Users\yihoi\AppData\Local\Temp\ipykernel_29052\4155643248.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  business_grouped = business_merged.groupby('CSDDGUID', as_index=False).agg(**agg_dict)


✅ Aggregated to 5161 CSDs with 11 category pairs


,CSDDGUID,All_Businesses,All_Employees,LANDAREA,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,...,212326,331210,111190,221320,339110,212316,337127,111994,114210,geometry
0,2021A00051001101,0,0,870.8928,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"MULTIPOLYGON (((8991051.954 2038839.069, 89910..."
1,2021A00051001105,1,7,1.0770,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"POLYGON ((9001504.369 2049856.597, 9001184.977..."
2,2021A00051001113,17,171,54.2130,1,24,1,24,1,24,...,0,0,0,0,0,0,0,0,0,"POLYGON ((8992311.551 2054468.074, 8994001.626..."
3,2021A00051001120,1,3,1.0729,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"POLYGON ((8985240.566 2028560.54, 8984967.974 ..."
4,2021A00051001124,29,199,742.3781,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"MULTIPOLYGON (((8995265.971 2098624.091, 89952..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5156,2021A00056208068,0,0,99.9450,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"MULTIPOLYGON (((5544792.883 3549525.897, 55447..."
5157,2021A00056208073,44,967,195.7789,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"MULTIPOLYGON (((5693598.966 3664907.42, 569358..."
5158,2021A00056208081,13,414,28.5462,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"MULTIPOLYGON (((6043125.089 3568426.163, 60431..."
5159,2021A00056208087,11,265,35.3819,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"MULTIPOLYGON (((6136803.711 3664909.323, 61367..."


# STEP 7: Processing it into Centroids for Counts and Choropleth for Rates

Since the earlier table shows the *weighted* numbers of directly exposed businesses and employees (by work location) together with *total* number of employees (by work location) for each affected NAICS code, the former is separated from the latter. The former needs to be processed into separate GDFs for centroids (to show counts) and choropleths (to show percentages)

In [15]:
# ============================================================
# SEPARATE CATEGORY DATA FROM NAICS DATA - DYNAMIC
# ============================================================

# Build excluded columns list dynamically
excluded_cols = ['All_Businesses', 'All_Employees', 'LANDAREA', 'geometry']
for prefix in all_category_prefixes:
    excluded_cols.extend([f'{prefix}_B', f'{prefix}_E'])

business_filter = business_grouped[['CSDDGUID'] + [col for col in business_grouped.columns if col in excluded_cols]].copy()

business_census = business_grouped[[col for col in business_grouped.columns if col not in excluded_cols]]

print(f"✅ business_filter has {len([c for c in excluded_cols if c.endswith('_B')])} category business columns")
print(f"   business_census has {len(business_census.columns)} NAICS columns")

✅ business_filter has 11 category business columns
   business_census has 311 NAICS columns


In [16]:
# ============================================================
# CREATE CENTROIDS - DYNAMIC CATEGORIES
# ============================================================

# Convert to GeoDataFrame for CENTROIDS (will use point geometry)
cent_gdf = gpd.GeoDataFrame(business_filter.copy(), geometry='geometry', crs = 'EPSG:3347')

# Set a point within each polygon for centroid display
cent_gdf = cent_gdf.drop(columns=['All_Businesses', 'All_Employees', 'LANDAREA'])
cent_gdf['geometry'] = cent_gdf.geometry.representative_point()
cent_gdf.set_geometry('geometry', inplace=True)
cent_gdf

,CSDDGUID,Auto_B,Auto_E,Alum_B,Alum_E,Steel_B,Steel_E,Cop_B,Cop_E,Lum_B,...,MHDV_E,LumOld_B,LumOld_E,LumNew_B,LumNew_E,CUSMA_B,CUSMA_E,Total_B,Total_E,geometry
0,2021A00051001101,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,POINT (9006087.523 2052813.611)
1,2021A00051001105,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,POINT (9000807.176 2050417.457)
2,2021A00051001113,1,24,1,24,1,24,0,0,0,...,0,0,0,0,0,1,24,1,24,POINT (8994180.445 2048230.129)
3,2021A00051001120,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,POINT (8984747.451 2028750.604)
4,2021A00051001124,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,3,1,3,POINT (8987903.538 2092846.593)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5156,2021A00056208068,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,POINT (5550011.303 3546406.473)
5157,2021A00056208073,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,15,1,15,POINT (5688775.424 3675693.766)
5158,2021A00056208081,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,POINT (6041286.124 3573109.546)
5159,2021A00056208087,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,POINT (6137793.914 3667840.123)


In [17]:
# Create choropleth from business_filter which still has POLYGON geometries
choro_cols = business_filter.copy()

# Calculate percentage rates for each category dynamically
for prefix in all_category_prefixes:
    # {prefix}_1 = % of businesses affected
    # {prefix}_2 = % of employees affected
    choro_cols[f'{prefix}_1'] = (
        choro_cols[f'{prefix}_B'] / choro_cols['All_Businesses']
    )
    choro_cols[f'{prefix}_2'] = (
        choro_cols[f'{prefix}_E'] / choro_cols['All_Employees']
    )

# Select only the columns we need
rate_cols = []
for prefix in all_category_prefixes:
    rate_cols.extend([f'{prefix}_1', f'{prefix}_2'])

choro_cols = choro_cols[['CSDDGUID'] + rate_cols + ['geometry']]

choro_gdf = gpd.GeoDataFrame(choro_cols, geometry='geometry', crs='EPSG:3347')

print(f"✅ Created choropleth with {len(rate_cols)} rate columns")
print(f"   Categories: {all_category_prefixes}")

choro_gdf

✅ Created choropleth with 22 rate columns
   Categories: ['Auto', 'Alum', 'Steel', 'Cop', 'Lum', 'Ene', 'MHDV', 'LumOld', 'LumNew', 'CUSMA', 'Total']


,CSDDGUID,Auto_1,Auto_2,Alum_1,Alum_2,Steel_1,Steel_2,Cop_1,Cop_2,Lum_1,...,MHDV_2,LumOld_1,LumOld_2,LumNew_1,LumNew_2,CUSMA_1,CUSMA_2,Total_1,Total_2,geometry
0,2021A00051001101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((8991051.954 2038839.069, 89910..."
1,2021A00051001105,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((9001504.369 2049856.597, 9001184.977..."
2,2021A00051001113,0.058824,0.140351,0.058824,0.140351,0.058824,0.140351,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.058824,0.140351,0.058824,0.140351,"POLYGON ((8992311.551 2054468.074, 8994001.626..."
3,2021A00051001120,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((8985240.566 2028560.54, 8984967.974 ..."
4,2021A00051001124,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.034483,0.015075,0.034483,0.015075,"MULTIPOLYGON (((8995265.971 2098624.091, 89952..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5156,2021A00056208068,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((5544792.883 3549525.897, 55447..."
5157,2021A00056208073,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.022727,0.015512,0.022727,0.015512,"MULTIPOLYGON (((5693598.966 3664907.42, 569358..."
5158,2021A00056208081,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,"MULTIPOLYGON (((6043125.089 3568426.163, 60431..."
5159,2021A00056208087,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,"MULTIPOLYGON (((6136803.711 3664909.323, 61367..."


# STEP 8: Finding Neighboring CSDs for Job Accessibility Analysis

Since employees often commute across CSD boundaries, we need to find neighboring CSDs to accurately estimate job accessibility from each residential location.

It is likely that while employees live close to their workplace, they do not live in the same CSD as they work in.  
  
StatsCan Census 2021 data shows a huge drop in the number of Canadians who travel more than 15km to their work vis-a-vis those who travel less than that distance to work.  
  
Thus, this cell creates a dictionary where for each CSD, it lists down, including itself, the CSD IDs within a 15km buffer around it (for small CSDs) or CSD IDs that are adjacent to it (for large CSDs). Small CSDs are defined as CSDs with an area less than (15km)^2 = 706 km^2.

In [18]:
# Copy from earlier CSD shapefile and ensure it's in projected CRS (EPSG:3347)
csds = csdc.to_crs("EPSG:3347").copy()

# Create a column to mark if CSD is "small" (<= 706 km2)
csds['is_small'] = csds['LANDAREA'] <= 706

# More aggressive simplification to prevent memory issues
# Different tolerance for small vs large CSDs
csds['geometry'] = csds.apply(
    lambda row: row['geometry'].simplify(1000 if row['is_small'] else 2000, preserve_topology=True),
    axis=1
)

# Build spatial index
csds_sindex = csds.sindex

# Prepare empty dictionary
csd_neighbors = {}

# Track overall time
start_time = time.time()

print(f"Processing {len(csds)} CSDs...")

# Process with error handling
for idx, row in tqdm(csds.iterrows(), total=len(csds)):
    csd_uid = row['CSDDGUID']
    geom = row['geometry']
    is_small = row['is_small']
    
    try:
        if is_small:
            # For small CSDs, use a different approach to avoid buffer issues
            # Get centroid and create a bounding box instead of buffer
            centroid = geom.centroid
            x, y = centroid.x, centroid.y
            # Create a 15km bounding box around centroid
            bbox = (x - 15000, y - 15000, x + 15000, y + 15000)
            
            # Spatial index query using bounding box
            possible_idx = list(csds_sindex.intersection(bbox))
            candidates = csds.iloc[possible_idx]
            
            # Filter by actual distance from centroid (more robust than buffer)
            matches = candidates[
                (candidates['CSDDGUID'] == csd_uid) |
                (candidates.geometry.centroid.distance(centroid) <= 15000)
            ]
        else:
            # Use adjacency for large CSDs
            bounds = geom.bounds
            # Expand bounds to catch touching geometries
            expanded_bounds = (bounds[0]-500, bounds[1]-500, bounds[2]+500, bounds[3]+500)
            
            possible_idx = list(csds_sindex.intersection(expanded_bounds))
            candidates = csds.iloc[possible_idx]
            
            # Use distance check instead of touches (more reliable)
            matches = candidates[
                (candidates['CSDDGUID'] == csd_uid) | 
                (candidates.geometry.distance(geom) < 500)
            ]
        
        csd_neighbors[csd_uid] = matches['CSDDGUID'].tolist()
        
    except Exception as e:
        # If there's an error, at minimum include the CSD itself
        print(f"\nWarning: Error processing {csd_uid}: {str(e)}")
        csd_neighbors[csd_uid] = [csd_uid]
    
    # Progress update every 500 CSDs
    if (idx + 1) % 500 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(csds) - idx - 1) / rate
        print(f"Processed {idx + 1}/{len(csds)} CSDs | Elapsed: {timedelta(seconds=int(elapsed))} | ETA: {timedelta(seconds=int(remaining))}")
        
        # Force garbage collection every 500 items
        gc.collect()

# Overall timing
total_elapsed = time.time() - start_time
print(f"\n✅ Total time taken: {timedelta(seconds=total_elapsed)}")
print(f"Average time per CSD: {total_elapsed/len(csds):.3f} seconds")

# Save the dictionary to disk
with open("csd_neighbors.json", "w") as f:
    json.dump(csd_neighbors, f)
    
print(f"Saved neighbors dictionary with {len(csd_neighbors)} CSDs")

# Verify completeness
print(f"Total CSDs with neighbors: {len(csd_neighbors)}")
print(f"Average neighbors per CSD: {sum(len(v) for v in csd_neighbors.values()) / len(csd_neighbors):.1f}")

Processing 5161 CSDs...


 11%|█         | 552/5161 [00:10<01:47, 42.88it/s] 

Processed 500/5161 CSDs | Elapsed: 0:00:10 | ETA: 0:01:40


 22%|██▏       | 1158/5161 [00:11<00:09, 423.21it/s]

Processed 1000/5161 CSDs | Elapsed: 0:00:11 | ETA: 0:00:47


...

Processed 4500/5161 CSDs | Elapsed: 0:00:33 | ETA: 0:00:04


 98%|█████████▊| 5073/5161 [00:44<00:01, 54.98it/s] 

Processed 5000/5161 CSDs | Elapsed: 0:00:44 | ETA: 0:00:01


100%|██████████| 5161/5161 [02:47<00:00, 30.83it/s]


✅ Total time taken: 0:02:47.399071
Average time per CSD: 0.032 seconds
Saved neighbors dictionary with 5161 CSDs
Total CSDs with neighbors: 5161
Average neighbors per CSD: 5.6


The dictionary is then used in conjunction with the *total* count of employees (by work location), as separated in Cell 13 above, to find out the likely number of jobs of each 6-digit NAICS, and total number of jobs, that are 'accessible' from each CSD --> going by the assumption of travel distance made by Canadians to go to work from Census 2021 data

In [19]:
# Ensure 'CSDDGUID' is the index for fast lookup
business_census_indexed = business_census.set_index('CSDDGUID')

# Debug: Check what's in business_census
print(f"business_census shape: {business_census.shape}")
print(f"business_census columns: {business_census.columns.tolist()}")
print(f"\nSample of business_census:")
print(business_census.head())
print(f"\nSum of all columns:")
print(business_census.sum())

# Normalize CSD IDs in business_census and ensure numeric job columns
business_census = business_census.copy()
business_census['CSDDGUID'] = business_census['CSDDGUID'].astype(str).str.strip()

business_census_indexed = (
    business_census
    .set_index('CSDDGUID')
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
)

# Normalize csd_neighbors IDs (keys + values)
csd_neighbors_norm = {
    str(k).strip(): [str(v).strip() for v in (vals or [])]
    for k, vals in csd_neighbors.items()
}

# Diagnostics: do neighbor IDs overlap business_census IDs?
idx_set = set(business_census_indexed.index)
keys_set = set(csd_neighbors_norm.keys())
vals_set = set(v for vals in csd_neighbors_norm.values() for v in vals)

print("business_census unique CSDs:", len(idx_set))
print("csd_neighbors keys:", len(keys_set))
print("csd_neighbors values (unique):", len(vals_set))
print("Overlap (neighbors keys ∩ business_census):", len(keys_set & idx_set))
print("Overlap (neighbors values ∩ business_census):", len(vals_set & idx_set))

if len(vals_set & idx_set) == 0:
    print("\n⚠️ No overlap found. Examples:")
    print("business_census sample IDs:", list(sorted(idx_set))[:5])
    print("csd_neighbors key sample IDs:", list(sorted(keys_set))[:5])
    print("csd_neighbors value sample IDs:", list(sorted(vals_set))[:5])

# Prepare list to collect results
aggregated_results = []

# Loop through CSD + its neighbors
for csd_id, neighbor_list in tqdm(csd_neighbors_norm.items()):
    # keep only neighbors that exist in business_census
    keep = list(idx_set.intersection(neighbor_list))
    rows = business_census_indexed.loc[keep] if keep else None

    summed = rows.sum() if rows is not None else business_census_indexed.iloc[:0].sum()
    result = summed.to_dict()
    result['CSDDGUID'] = csd_id
    aggregated_results.append(result)

jobs = pd.DataFrame(aggregated_results)

print(f"\njobs shape: {jobs.shape}")
print("Total jobs (sum over all NAICS cols):", jobs.drop(columns='CSDDGUID').to_numpy().sum())

jobs

business_census shape: (5161, 311)
business_census columns: ['CSDDGUID', '327310', '212299', '333519', '332999', '314990', '332321', '324190', '336310', '311352', '331529', '332619', '337920', '332210', '311515', '337110', '325610', '311230', '314910', '312210', '311221', '332991', '325189', '113311', '321211', '311214', '327120', '315990', '335229', '327390', '112920', '333246', '111130', '336340', '111422', '333910', '112930', '212396', '212393', '111411', '332611', '316990', '339990', '311619', '313110', '325110', '332420', '333247', '339910', '331490', '332314', '322211', '112120', '336360', '334220', '335930', '212315', '111930', '331317', '334290', '333920', '311814', '339950', '333619', '335223', '211110', '221111', '335210', '337214', '212317', '325991', '212323', '111999', '331511', '212114', '315250', '111412', '323120', '112210', '332410', '333310', '212395', '111940', '325910', '336215', '332431', '114114', '326160', '312130', '321114', '311920', '311340', '311614', '327330

100%|██████████| 5161/5161 [00:02<00:00, 2033.08it/s]



jobs shape: (5161, 311)
Total jobs (sum over all NAICS cols): 10662207


,327310,212299,333519,332999,314990,332321,324190,336310,311352,331529,...,212326,331210,111190,221320,339110,212316,337127,111994,114210,CSDDGUID
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2021A00051001101
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2021A00051001105
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2021A00051001113
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2021A00051001120
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2021A00051001124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5156,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2021A00056208068
5157,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2021A00056208073
5158,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2021A00056208081
5159,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2021A00056208087


Calculate the rate of directly exposed jobs (6-digit NAICS) to total jobs in each industry (1-digit NAICS) within each CSD

In [20]:
jobs_rate = jobs.copy()

cola = [col for col in jobs.columns if col != 'CSDDGUID']

print(f"Number of NAICS columns: {len(cola)}")
print(f"Sample NAICS columns: {cola[:10]}")

# Calculate summed groups by first digit of column name
jobs_rate['Sum1'] = jobs_rate[[col for col in cola if col.startswith('1')]].sum(axis=1)
jobs_rate['Sum2'] = jobs_rate[[col for col in cola if col.startswith('2')]].sum(axis=1)
jobs_rate['Sum3'] = jobs_rate[[col for col in cola if col.startswith('3')]].sum(axis=1)

print(f"\nSum1 range: {jobs_rate['Sum1'].min()} to {jobs_rate['Sum1'].max()}")
print(f"Sum2 range: {jobs_rate['Sum2'].min()} to {jobs_rate['Sum2'].max()}")
print(f"Sum3 range: {jobs_rate['Sum3'].min()} to {jobs_rate['Sum3'].max()}")

# Compute share per column with proper division by zero handling
for col in cola:
    col_rate = f'{col}_R'
    if col.startswith('1'):
        jobs_rate[col_rate] = np.where(
            jobs_rate['Sum1'] > 0,
            jobs_rate[col] / jobs_rate['Sum1'],
            0
        )
    elif col.startswith('2'):
        jobs_rate[col_rate] = np.where(
            jobs_rate['Sum2'] > 0,
            jobs_rate[col] / jobs_rate['Sum2'],
            0
        )
    elif col.startswith('3'):
        jobs_rate[col_rate] = np.where(
            jobs_rate['Sum3'] > 0,
            jobs_rate[col] / jobs_rate['Sum3'],
            0
        )
    else:
        jobs_rate[col_rate] = 0  # fallback in case of unexpected prefix

# Final filtered DataFrame: only CSDDGUID and the *_R columns
rate_cols = [f'{col}_R' for col in cola]
jobs_rate = jobs_rate[['CSDDGUID'] + rate_cols]

jobs_rate = jobs_rate.fillna(0)

# Debug: Check the results
print(f"\nNon-zero rates count:")
for prefix in ['1', '2', '3']:
    prefix_cols = [c for c in rate_cols if c[0] == prefix]
    if prefix_cols:
        non_zero = (jobs_rate[prefix_cols] > 0).sum().sum()
        total = len(prefix_cols) * len(jobs_rate)
        print(f"  {prefix}xx NAICS: {non_zero}/{total} non-zero values")

print(f"\nSample of jobs_rate:")
print(jobs_rate.head())

jobs_rate.to_csv('trail6_view_csd.csv')

jobs_rate

Number of NAICS columns: 310
Sample NAICS columns: ['327310', '212299', '333519', '332999', '314990', '332321', '324190', '336310', '311352', '331529']

Sum1 range: 0 to 14690
Sum2 range: 0 to 15616
Sum3 range: 0 to 113429


C:\Users\yihoi\AppData\Local\Temp\ipykernel_29052\3941140973.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  jobs_rate[col_rate] = np.where(
C:\Users\yihoi\AppData\Local\Temp\ipykernel_29052\3941140973.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  jobs_rate[col_rate] = np.where(
C:\Users\yihoi\AppData\Local\Temp\ipykernel_29052\3941140973.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all column


Non-zero rates count:
  1xx NAICS: 41996/221923 non-zero values
  2xx NAICS: 4534/144508 non-zero values
  3xx NAICS: 84165/1233479 non-zero values

Sample of jobs_rate:
           CSDDGUID  327310_R  212299_R  333519_R  332999_R  314990_R  \
0  2021A00051001101       0.0       0.0       0.0       0.0       0.0   
1  2021A00051001105       0.0       0.0       0.0       0.0       0.0   
2  2021A00051001113       0.0       0.0       0.0       0.0       0.0   
3  2021A00051001120       0.0       0.0       0.0       0.0       0.0   
4  2021A00051001124       0.0       0.0       0.0       0.0       0.0   

   332321_R  324190_R  336310_R  311352_R  ...  111419_R  212326_R  331210_R  \
0       0.0       0.0       0.0       0.0  ...       0.0       0.0       0.0   
1       0.0       0.0       0.0       0.0  ...       0.0       0.0       0.0   
2       0.0       0.0       0.0       0.0  ...       0.0       0.0       0.0   
3       0.0       0.0       0.0       0.0  ...       0.0       0.0    

,CSDDGUID,327310_R,212299_R,333519_R,332999_R,314990_R,332321_R,324190_R,336310_R,311352_R,...,111419_R,212326_R,331210_R,111190_R,221320_R,339110_R,212316_R,337127_R,111994_R,114210_R
0,2021A00051001101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2021A00051001105,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2021A00051001113,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2021A00051001120,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2021A00051001124,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5156,2021A00056208068,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5157,2021A00056208073,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5158,2021A00056208081,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5159,2021A00056208087,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# STEP 9: Applying Jobs Weight to Census Data

Census 2021 Data reports residents' occupational NAICS code at the two-digit level

In [21]:
# Use the correct CSD-level census file
census = pd.read_csv('input-data/large_size_data/98-401-X2021003_English_CSV_data.csv', encoding='latin1')

print(f'Census raw shape: {census.shape}')

# Filter to Census subdivision level only
census = census[census['GEO_LEVEL'] == 'Census subdivision'].copy()
print(f'After GEO_LEVEL filter (Census subdivision): {census.shape}')

# Filter by Characteristic IDs (Employment by Industry at 2-digit NAICS)
census = census[census['CHARACTERISTIC_ID'].isin([2259, 2262, 2263, 2266])].copy()
print(f'After CHARACTERISTIC_ID filter: {census.shape}')

# Create CSDDGUID directly from DGUID column
census['CSDDGUID'] = census['DGUID'].astype(str)

# Province extraction from DGUID (characters 10-11, 0-indexed as 9:11)
census['ProvinceCode'] = census['CSDDGUID'].str[9:11].astype(int)
census['Province'] = census['ProvinceCode'].map(province_code)

# Characteristic Name cleanup to get 2-digit NAICS codes
census['CHARACTERISTIC_NAME'] = (
    census['CHARACTERISTIC_NAME']
    .astype(str)
    .str.replace(' ', '', regex=False)
    .str[:2]
)

census = census[['CSDDGUID', 'Province', 'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL']]

# Pivot
census_pivot = census.pivot_table(
    index=['CSDDGUID', 'Province'],
    columns='CHARACTERISTIC_NAME',
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

print(f"Census pivot shape: {census_pivot.shape}")
print(f"Census pivot columns: {census_pivot.columns.tolist()}")
print(f"Sample IDs (Census): {census_pivot['CSDDGUID'].head().tolist()}")

# --- Diagnostics ---
common_ids = set(census_pivot['CSDDGUID']).intersection(set(jobs_rate['CSDDGUID']))
print(f"Overlap between Census and Jobs Rate IDs: {len(common_ids)}")

if len(common_ids) == 0:
    print("⚠️ No overlap found! Check ID formats.")
    print("Sample IDs (Jobs Rate):", jobs_rate['CSDDGUID'].head().tolist())
else:
    print(f"✅ IDs overlap: {len(common_ids)} CSDs matched")

census_pivot.head()

Census raw shape: (3049329, 23)
After GEO_LEVEL filter (Census subdivision): (2649417, 23)
After CHARACTERISTIC_ID filter: (4028, 23)
Census pivot shape: (937, 6)
Census pivot columns: ['CSDDGUID', 'Province', '11', '21', '31', 'To']
Sample IDs (Census): ['2021A00051001472', '2021A00051001485', '2021A00051001504', '2021A00051001505', '2021A00051001507']
Overlap between Census and Jobs Rate IDs: 937
✅ IDs overlap: 937 CSDs matched


CHARACTERISTIC_NAME,CSDDGUID,Province,11,21,31,To
0,2021A00051001472,NL,15.0,30.0,45.0,1105.0
1,2021A00051001485,NL,160.0,495.0,540.0,14500.0
2,2021A00051001504,NL,80.0,180.0,130.0,4640.0
3,2021A00051001505,NL,20.0,25.0,30.0,985.0
4,2021A00051001507,NL,30.0,45.0,25.0,955.0


Thus the weights from Step 8 is used to estimate how many employees (by primary residence) are working in industries directly exposed to tariffs

In [22]:
# Merge on CSDDGUID to align both datasets - use 'left' to keep all census data
merged = census_pivot.merge(jobs_rate, on='CSDDGUID', how='left')

print(f"Merged shape: {merged.shape}")
print(f"Available columns: {merged.columns.tolist()}")
print(f"Sample 'To' values: {merged['To'].head()}")
print(f"To column stats: min={merged['To'].min()}, max={merged['To'].max()}, mean={merged['To'].mean()}")

# Start building the adjusted DataFrame
adjusted_jobs = merged[['CSDDGUID', 'Province', 'To']].copy()

# Compute adjusted values
for col in cola:
    rate_col = f'{col}_R'
    prefix = col[:2]

    if prefix in ['21', '22']:
        source_col = '21'
    elif prefix in ['31', '32', '33']:
        source_col = '31'
    else:
        source_col = prefix

    # Check if both rate and source columns exist
    if rate_col in merged.columns and source_col in merged.columns:
        adjusted_jobs[col] = np.ceil(merged[rate_col].fillna(0) * merged[source_col].fillna(0))
    else:
        # If either column is missing, assume 0
        adjusted_jobs[col] = 0
        if rate_col not in merged.columns:
            print(f"⚠️ Missing rate column: {rate_col}")
        if source_col not in merged.columns:
            print(f"⚠️ Missing source column: {source_col}")

adjusted_jobs['Sum'] = adjusted_jobs.drop(columns=['CSDDGUID', 'Province', 'To']).sum(axis=1)

adjusted_jobs = adjusted_jobs.fillna(0)

adjusted_jobs['Province'] = adjusted_jobs['Province'].mask(
    adjusted_jobs['Province'].isin([0, np.nan]),
    adjusted_jobs['CSDDGUID'].str[9:11].astype(int).map(province_code)
)

print(f"\nAdjusted jobs shape: {adjusted_jobs.shape}")
print(f"Total adjusted jobs sum: {adjusted_jobs['Sum'].sum()}")
print(f"Rows with non-zero To: {(adjusted_jobs['To'] > 0).sum()}")
print(f"Rows with non-zero Sum: {(adjusted_jobs['Sum'] > 0).sum()}")

adjusted_jobs

Merged shape: (937, 316)
Available columns: ['CSDDGUID', 'Province', '11', '21', '31', 'To', '327310_R', '212299_R', '333519_R', '332999_R', '314990_R', '332321_R', '324190_R', '336310_R', '311352_R', '331529_R', '332619_R', '337920_R', '332210_R', '311515_R', '337110_R', '325610_R', '311230_R', '314910_R', '312210_R', '311221_R', '332991_R', '325189_R', '113311_R', '321211_R', '311214_R', '327120_R', '315990_R', '335229_R', '327390_R', '112920_R', '333246_R', '111130_R', '336340_R', '111422_R', '333910_R', '112930_R', '212396_R', '212393_R', '111411_R', '332611_R', '316990_R', '339990_R', '311619_R', '313110_R', '325110_R', '332420_R', '333247_R', '339910_R', '331490_R', '332314_R', '322211_R', '112120_R', '336360_R', '334220_R', '335930_R', '212315_R', '111930_R', '331317_R', '334290_R', '333920_R', '311814_R', '339950_R', '333619_R', '335223_R', '211110_R', '221111_R', '335210_R', '337214_R', '212317_R', '325991_R', '212323_R', '111999_R', '331511_R', '212114_R', '315250_R', '111412

C:\Users\yihoi\AppData\Local\Temp\ipykernel_29052\424694594.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adjusted_jobs[col] = np.ceil(merged[rate_col].fillna(0) * merged[source_col].fillna(0))
C:\Users\yihoi\AppData\Local\Temp\ipykernel_29052\424694594.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adjusted_jobs[col] = np.ceil(merged[rate_col].fillna(0) * merged[source_col].fillna(0))
C:\Users\yihoi\AppData\Local\Temp\ipykernel_29052\424694594.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usua

,CSDDGUID,Province,To,327310,212299,333519,332999,314990,332321,324190,...,212326,331210,111190,221320,339110,212316,337127,111994,114210,Sum
0,2021A00051001472,NL,1105.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45.0
1,2021A00051001485,NL,14500.0,4.0,0.0,0.0,20.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,17.0,0.0,4.0,0.0,0.0,1217.0
2,2021A00051001504,NL,4640.0,1.0,0.0,0.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,4.0,0.0,1.0,0.0,3.0,409.0
3,2021A00051001505,NL,985.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,80.0
4,2021A00051001507,NL,955.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,106.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
932,2021A00056001055,YK,320.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
933,2021A00056001058,YK,460.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
934,2021A00056001059,YK,920.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,154.0
935,2021A00056001060,YK,160.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# STEP 10: Applying Export Weights to the Census Data

Export weights from Step 4 are applied to account for regional differences

In [23]:
# Step 1: Melt to long format
long_weight = weight_naics.melt(id_vars='NAICS', var_name='Province', value_name='Rate')

long_weight['NAICS'] = long_weight['NAICS'].astype(str) + '_r'

# Step 2: Pivot to wide format
weight_naics_pivot = long_weight.pivot(index=['Province'], columns='NAICS', values='Rate').reset_index()

weight_naics_pivot

NAICS,Province,111110_r,111120_r,111130_r,111140_r,111150_r,111160_r,111190_r,111211_r,111219_r,...,337215_r,337910_r,337920_r,339110_r,339910_r,339920_r,339930_r,339940_r,339950_r,339990_r
0,AL,0.024138,0.034204,0.0,0.032376,0.370000,0.000000,0.117974,0.0,0.026535,...,0.987325,0.514049,0.213624,0.532096,0.529703,0.663320,0.368632,0.720095,0.206830,0.425631
1,BC,0.000000,0.028733,0.0,0.028748,0.243865,0.000000,0.156439,0.0,0.000031,...,0.886988,0.186107,0.213990,0.590677,0.746073,0.587708,0.255254,0.691042,0.265313,0.269257
2,MB,0.064938,0.094353,0.0,0.030494,0.370000,0.000000,0.264963,0.0,0.127486,...,0.995474,0.790000,0.131253,0.450144,0.332359,0.582863,0.337235,0.859608,0.151730,0.573977
3,NB,0.620000,0.124930,0.0,0.620000,0.000000,0.000000,0.000000,0.0,0.004048,...,0.998284,0.790000,0.205144,0.849244,0.890000,0.243381,0.420000,0.831198,0.699527,0.330572
4,NL,0.000000,0.620000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.956908,0.000000,0.000000,0.416990,0.268019,0.079019,0.358952,0.000000,0.000000,0.132630
5,NS,0.000000,0.000000,0.0,0.620000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.914146,0.000000,0.000000,0.703223,0.786739,0.706480,0.397013,0.049719,0.149934,0.471560
6,NU,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.870000,0.890000,0.000000,0.420000,0.000000,0.150000,0.240000
7,NWT,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.890000,0.000000,0.000000,0.000000,0.000000,0.000000
8,ON,0.057076,0.371635,0.0,0.032567,0.074227,0.315716,0.206290,0.0,0.001356,...,0.963866,0.728146,0.187830,0.555835,0.907276,0.653166,0.248730,0.732703,0.562703,0.597974
9,PEI,0.000000,0.620000,0.0,0.000000,0.000000,0.000000,0.370000,0.0,0.000000,...,0.106944,0.000000,0.000000,0.000000,0.890000,0.227019,0.000000,0.000000,0.150000,0.485059


In [24]:
# ============================================================
# CALCULATE CENSUS BY TARIFFS - DYNAMIC CATEGORIES
# ============================================================

# Step 1: Merge adjusted_jobs with weight_naics_pivot on Province
census_byjobs = adjusted_jobs.merge(weight_naics_pivot, on='Province', how='left')

# Step 2: Multiply each column by its corresponding rate
for col in cola:
    rate_col = f'{col}_r'
    
    census_byjobs[col] = np.ceil(census_byjobs[col] * census_byjobs[rate_col])

# Step 3: Keep only CSDDGUID and updated values
census_byjobs = census_byjobs[['CSDDGUID', 'To'] + cola]

# Define output dictionary
grouped_data = {
    'CSDDGUID': census_byjobs['CSDDGUID'],  # retain CSD ID
    'Census': census_byjobs['To'],
}

# Add _C columns for each category dynamically
for prefix, naics_set in category_naics.items():
    matching_cols = [col for col in cola if col in naics_set]
    if matching_cols:
        grouped_data[f'{prefix}_C'] = census_byjobs[matching_cols].sum(axis=1)
    else:
        grouped_data[f'{prefix}_C'] = 0

# Create final grouped DataFrame
census_bytariffs = pd.DataFrame(grouped_data)

print(f"✅ Created census_bytariffs with {len([k for k in grouped_data if k.endswith('_C')])} category columns")

census_bytariffs

✅ Created census_bytariffs with 11 category columns


,CSDDGUID,Census,Auto_C,Alum_C,Steel_C,Cop_C,Lum_C,Ene_C,MHDV_C,LumOld_C,LumNew_C,CUSMA_C,Total_C
0,2021A00051001472,1105.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,6.0,6.0
1,2021A00051001485,14500.0,8.0,23.0,20.0,3.0,0,8.0,0.0,28.0,99.0,333.0,333.0
2,2021A00051001504,4640.0,4.0,9.0,7.0,1.0,0,24.0,0.0,6.0,23.0,126.0,126.0
3,2021A00051001505,985.0,2.0,2.0,1.0,0.0,0,3.0,0.0,0.0,16.0,28.0,28.0
4,2021A00051001507,955.0,2.0,2.0,1.0,0.0,0,6.0,0.0,0.0,14.0,31.0,31.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
932,2021A00056001055,320.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
933,2021A00056001058,460.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
934,2021A00056001059,920.0,1.0,2.0,0.0,0.0,0,8.0,0.0,2.0,2.0,23.0,23.0
935,2021A00056001060,160.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0


# STEP 11: Processing and adding the data to Choropleth and Centroid GDFs

Add the data on employees (by primary residence) to the GDFs produced in Step 7

In [25]:
# ============================================================
# MERGE CENSUS DATA WITH CENTROIDS - DYNAMIC CATEGORIES
# ============================================================

centroids = cent_gdf.merge(census_bytariffs, on='CSDDGUID', how='left')

# Build column list dynamically
centroid_cols = ['CSDDGUID']
for prefix in all_category_prefixes:
    centroid_cols.extend([f'{prefix}_B', f'{prefix}_E', f'{prefix}_C'])
centroid_cols.append('geometry')

centroids = centroids[centroid_cols]
centroids = centroids.to_crs('EPSG:4326')
centroids.to_file('centroids_csd.geojson', driver='GeoJSON')
centroids.to_csv("centroids_csd.csv", index=False)

print(f"✅ Saved centroids with {len(all_category_prefixes)} categories x 3 metrics (B, E, C)")

centroids

✅ Saved centroids with 11 categories x 3 metrics (B, E, C)


,CSDDGUID,Auto_B,Auto_E,Auto_C,Alum_B,Alum_E,Alum_C,Steel_B,Steel_E,Steel_C,...,LumNew_B,LumNew_E,LumNew_C,CUSMA_B,CUSMA_E,CUSMA_C,Total_B,Total_E,Total_C,geometry
0,2021A00051001101,0,0,NaN,0,0,NaN,0,0,NaN,...,0,0,NaN,0,0,NaN,0,0,NaN,POINT (-53.18157 46.70096)
1,2021A00051001105,0,0,NaN,0,0,NaN,0,0,NaN,...,0,0,NaN,0,0,NaN,0,0,NaN,POINT (-53.25545 46.71029)
2,2021A00051001113,1,24,NaN,1,24,NaN,1,24,NaN,...,0,0,NaN,1,24,NaN,1,24,NaN,POINT (-53.34216 46.72793)
3,2021A00051001120,0,0,NaN,0,0,NaN,0,0,NaN,...,0,0,NaN,0,0,NaN,0,0,NaN,POINT (-53.58585 46.63277)
4,2021A00051001124,0,0,NaN,0,0,NaN,0,0,NaN,...,0,0,NaN,1,3,NaN,1,3,NaN,POINT (-53.07768 47.08655)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5156,2021A00056208068,0,0,NaN,0,0,NaN,0,0,NaN,...,0,0,NaN,0,0,NaN,0,0,NaN,POINT (-107.82111 67.68532)
5157,2021A00056208073,0,0,NaN,0,0,NaN,0,0,NaN,...,0,0,NaN,1,15,NaN,1,15,NaN,POINT (-105.17569 69.13829)
5158,2021A00056208081,0,0,NaN,0,0,NaN,0,0,NaN,...,0,0,NaN,0,0,NaN,0,0,NaN,POINT (-95.88322 68.64144)
5159,2021A00056208087,0,0,NaN,0,0,NaN,0,0,NaN,...,0,0,NaN,0,0,NaN,0,0,NaN,POINT (-93.50447 69.55288)


In [26]:
# ============================================================
# CALCULATE PERCENTAGES FOR CHOROPLETH - DYNAMIC CATEGORIES
# ============================================================

perc_bytariffs = census_bytariffs.copy()

# Calculate _3 (% of census population in affected jobs) for each category
for prefix in all_category_prefixes:
    perc_bytariffs[f'{prefix}_3'] = (
        perc_bytariffs[f'{prefix}_C'] / perc_bytariffs['Census']
    )

# Select only the percentage columns
perc_cols = ['CSDDGUID'] + [f'{prefix}_3' for prefix in all_category_prefixes]
perc_bytariffs = perc_bytariffs[perc_cols]

# Clip values to max 1 (100%) for CUSMA and Total
perc_bytariffs['CUSMA_3'] = perc_bytariffs['CUSMA_3'].clip(upper=1)
perc_bytariffs['Total_3'] = perc_bytariffs['Total_3'].clip(upper=1)

print(f"✅ Calculated _3 percentages for {len(all_category_prefixes)} categories")

perc_bytariffs

✅ Calculated _3 percentages for 11 categories


,CSDDGUID,Auto_3,Alum_3,Steel_3,Cop_3,Lum_3,Ene_3,MHDV_3,LumOld_3,LumNew_3,CUSMA_3,Total_3
0,2021A00051001472,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.005430,0.005430
1,2021A00051001485,0.000552,0.001586,0.001379,0.000207,0.0,0.000552,0.0,0.001931,0.006828,0.022966,0.022966
2,2021A00051001504,0.000862,0.001940,0.001509,0.000216,0.0,0.005172,0.0,0.001293,0.004957,0.027155,0.027155
3,2021A00051001505,0.002030,0.002030,0.001015,0.000000,0.0,0.003046,0.0,0.000000,0.016244,0.028426,0.028426
4,2021A00051001507,0.002094,0.002094,0.001047,0.000000,0.0,0.006283,0.0,0.000000,0.014660,0.032461,0.032461
...,...,...,...,...,...,...,...,...,...,...,...,...
932,2021A00056001055,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
933,2021A00056001058,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
934,2021A00056001059,0.001087,0.002174,0.000000,0.000000,0.0,0.008696,0.0,0.002174,0.002174,0.025000,0.025000
935,2021A00056001060,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000


In [27]:
# ============================================================
# MERGE ALL DATA INTO FINAL CHOROPLETH - DYNAMIC CATEGORIES
# ============================================================

# Use 'left' merge to preserve all geometries from choro_gdf
choropleth = choro_gdf.merge(perc_bytariffs, on='CSDDGUID', how='left')

print(f"Choropleth shape after merge: {choropleth.shape}")

print(f"Geometry column type: {type(choropleth['geometry'].iloc[0]) if len(choropleth) > 0 else 'Empty'}")

print(f"Non-null geometries: {choropleth['geometry'].notna().sum()}")

print(f"CRS: {choropleth.crs}")

# Build final column list dynamically
final_cols = ['CSDDGUID']
for prefix in all_category_prefixes:
    final_cols.extend([f'{prefix}_1', f'{prefix}_2', f'{prefix}_3'])
final_cols.append('geometry')

choropleth = choropleth[final_cols]
print(f"Final choropleth shape: {choropleth.shape}")
print(f"Columns: {list(choropleth.columns)}")

# Ensure it's still a GeoDataFrame before CRS conversion
if not isinstance(choropleth, gpd.GeoDataFrame):
    print("⚠️ Converting back to GeoDataFrame")
    choropleth = gpd.GeoDataFrame(choropleth, geometry='geometry', crs='EPSG:3347')

# Convert to WGS84 (EPSG:4326) for better compatibility with mapping software
print(f"Converting from {choropleth.crs} to EPSG:4326...")
choropleth = choropleth.to_crs('EPSG:4326')
print(f"✅ CRS after conversion: {choropleth.crs}")

Choropleth shape after merge: (5161, 35)
Geometry column type: <class 'shapely.geometry.multipolygon.MultiPolygon'>
Non-null geometries: 5161
CRS: EPSG:3347
Final choropleth shape: (5161, 35)
Columns: ['CSDDGUID', 'Auto_1', 'Auto_2', 'Auto_3', 'Alum_1', 'Alum_2', 'Alum_3', 'Steel_1', 'Steel_2', 'Steel_3', 'Cop_1', 'Cop_2', 'Cop_3', 'Lum_1', 'Lum_2', 'Lum_3', 'Ene_1', 'Ene_2', 'Ene_3', 'MHDV_1', 'MHDV_2', 'MHDV_3', 'LumOld_1', 'LumOld_2', 'LumOld_3', 'LumNew_1', 'LumNew_2', 'LumNew_3', 'CUSMA_1', 'CUSMA_2', 'CUSMA_3', 'Total_1', 'Total_2', 'Total_3', 'geometry']
Converting from EPSG:3347 to EPSG:4326...
✅ CRS after conversion: EPSG:4326


In [28]:
# Delete old files if they exist to ensure clean save
import os
for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg']:
    try:
        os.remove(f'choropleth_csd{ext}')
    except FileNotFoundError:
        pass

# Verify geometries before saving
print(f"Geometry check before save:")
print(f"  - Total rows: {len(choropleth)}")
print(f"  - Valid geometries: {choropleth.geometry.is_valid.sum()}")
print(f"  - Geometry types: {choropleth.geometry.type.unique()}")
print(f"  - CRS: {choropleth.crs}")

# Save with EPSG:4326 coordinates
choropleth.to_file('choropleth_csd.geojson', driver='GeoJSON')
choropleth.to_file('choropleth_csd.shp', driver='ESRI Shapefile')
print(f"✅ Files saved with CRS: {choropleth.crs}")

Geometry check before save:
  - Total rows: 5161
  - Valid geometries: 5107
  - Geometry types: ['MultiPolygon' 'Polygon']
  - CRS: EPSG:4326
✅ Files saved with CRS: EPSG:4326


In [29]:
choropleth.head()

,CSDDGUID,Auto_1,Auto_2,Auto_3,Alum_1,Alum_2,Alum_3,Steel_1,Steel_2,Steel_3,...,LumNew_1,LumNew_2,LumNew_3,CUSMA_1,CUSMA_2,CUSMA_3,Total_1,Total_2,Total_3,geometry
0,2021A00051001101,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-53.44465 46.67495, -53.44474 ..."
1,2021A00051001105,0.000000,0.000000,NaN,0.000000,0.000000,NaN,0.000000,0.000000,NaN,...,0.0,0.0,NaN,0.000000,0.000000,NaN,0.000000,0.000000,NaN,"POLYGON ((-53.25217 46.70264, -53.25709 46.702..."
2,2021A00051001113,0.058824,0.140351,NaN,0.058824,0.140351,NaN,0.058824,0.140351,NaN,...,0.0,0.0,NaN,0.058824,0.140351,NaN,0.058824,0.140351,NaN,"POLYGON ((-53.31601 46.78311, -53.31595 46.756..."
3,2021A00051001120,0.000000,0.000000,NaN,0.000000,0.000000,NaN,0.000000,0.000000,NaN,...,0.0,0.0,NaN,0.000000,0.000000,NaN,0.000000,0.000000,NaN,"POLYGON ((-53.58199 46.62889, -53.58608 46.629..."
4,2021A00051001124,0.000000,0.000000,NaN,0.000000,0.000000,NaN,0.000000,0.000000,NaN,...,0.0,0.0,NaN,0.034483,0.015075,NaN,0.034483,0.015075,NaN,"MULTIPOLYGON (((-52.95556 47.0911, -52.95564 4..."


In [30]:
choropleth.drop(columns="geometry").to_csv("choropleth_csd.csv", index=False)

# CSV Trails

for double-checking purposes

In [31]:
business_grouped.drop(columns='geometry').to_csv('trail_csd.csv', index=False)
business_filter.drop(columns='geometry').to_csv('trail2_csd.csv', index=False)
business_census.to_csv('trail3_csd.csv', index=False)

In [32]:
choro_cols.drop(columns='geometry').to_csv('trail4_csd.csv', index=False)

In [33]:
jobs.to_csv('trail5_csd.csv')
jobs_rate.to_csv('trail6_csd.csv')
adjusted_jobs.to_csv('trail7_csd.csv')